# Fine-tune Llama-3.2-3B to sound like me (Unsloth + 4-bit QLoRA)

**Setup:** Runtime → Change runtime type → **T4 GPU**, then run the cells top to bottom.

You'll need your `.jsonl` file (one `{"messages": [...]}` object per line, user + assistant turns).

## 1. Install

In [ ]:
%%capture
!pip install unsloth
# If you hit weird version/OOM errors on a T4, restart the runtime and try:
# !pip install unsloth==2026.6.7 unsloth_zoo==2026.6.7

In [ ]:
import re, random
from collections import Counter, defaultdict
from datasets import Dataset, load_dataset

# ============ SETTINGS (edit these) ============
OVERUSED       = ["tuff"]   # words the model overuses. Run once, look at the printed top words, add more if needed
OVERUSED_KEEP  = 0.3        # keep this fraction of examples whose reply contains an overused word
FILLER         = {"lol", "ok", "ya", "yes", "no", "k", "haha", "hmm", "yeah", "yep", "sure", "okay", "bro"}
FILLER_KEEP    = 0.25       # keep this fraction of examples whose whole reply is just a filler word
MAX_SAME_REPLY = 2          # keep at most this many examples with the exact same reply
MIX_GENERIC    = True       # regularisation: mix in some generic chat data
GENERIC_FRAC   = 0.10       # generic examples = this fraction of your data size
SEED           = 3407
# ===============================================

rng = random.Random(SEED)

def reply_text(ex):
    return ex["messages"][-1]["content"]

def words(text):
    return re.findall(r"[a-z']+", text.lower())

STOP = set("the a an and or but to of in on at for is it i u you me my we he she they that this was are be do not so if with just like im its dont have get got what when how can will would".split())

def top_words(data, n=30):
    c = Counter()
    for ex in data:
        c.update(w for w in words(reply_text(ex)) if w not in STOP and len(w) > 2)
    return c.most_common(n)

print(f"Start: {len(rows)} examples")
print("Top words in your replies BEFORE:", top_words(rows))

overused = {w.lower() for w in OVERUSED}
cleaned, seen = [], defaultdict(int)
dropped = Counter()

for ex in rows:
    r = reply_text(ex).strip()
    norm = " ".join(words(r)) or r.lower()   # emoji/number-only replies are kept
    if not r:
        dropped["empty"] += 1
        continue
    # 1. filler-only replies: keep a fraction
    if norm in FILLER and rng.random() > FILLER_KEEP:
        dropped["filler"] += 1
        continue
    # 2. replies containing an overused word: keep a fraction
    if overused & set(words(r)) and rng.random() > OVERUSED_KEEP:
        dropped["overused word"] += 1
        continue
    # 3. cap exact-duplicate replies
    if seen[norm] >= MAX_SAME_REPLY:
        dropped["duplicate reply"] += 1
        continue
    seen[norm] += 1
    cleaned.append(ex)

print(f"\nAfter cleaning: {len(cleaned)} examples. Dropped: {dict(dropped)}")
print("Top words AFTER:", top_words(cleaned))

# ---- regularisation: mix in a little generic chat data ----
final = list(cleaned)
if MIX_GENERIC:
    try:
        n_generic = int(len(cleaned) * GENERIC_FRAC)
        gen = load_dataset("mlabonne/FineTome-100k", split="train[:6000]").shuffle(seed=SEED)
        added = 0
        for g in gen:
            conv = g["conversations"][:2]          # first question + answer only
            if len(conv) == 2 and conv[0]["from"] == "human" and conv[1]["from"] == "gpt":
                if len(conv[0]["value"]) + len(conv[1]["value"]) < 1200:   # keep short so it fits seq length
                    final.append({"messages": [
                        {"role": "user", "content": conv[0]["value"]},
                        {"role": "assistant", "content": conv[1]["value"]},
                    ]})
                    added += 1
                    if added >= n_generic:
                        break
        print(f"\nMixed in {added} generic examples")
    except Exception as e:
        print("\nSkipped generic mix-in:", type(e).__name__, str(e)[:100])

dataset = Dataset.from_list(final).shuffle(seed=SEED)
print(f"Final dataset: {len(dataset)} examples")

## 2. Upload and check your data

In [ ]:
from google.colab import files
import os

uploaded = files.upload()          # pick your .jsonl file
DATA_PATH = list(uploaded.keys())[0]
print("Using:", DATA_PATH, f"({os.path.getsize(DATA_PATH)/1e6:.2f} MB)")

In [ ]:
import json
from datasets import Dataset

rows, bad = [], 0
with open(DATA_PATH, encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            msgs = obj["messages"]
            assert msgs[-1]["role"] == "assistant" and msgs[0]["role"] in ("user", "system")
            rows.append({"messages": msgs})
        except Exception as e:
            bad += 1
            print(f"Skipping line {i}: {type(e).__name__}")

print(f"Loaded {len(rows)} examples, skipped {bad} bad lines")
dataset = Dataset.from_list(rows).shuffle(seed=3407)
print(dataset[0]["messages"])

## 3. Load the 4-bit model and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024   # your examples are short; raise if you see truncation

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,            # auto: fp16 on T4 (no bf16 support)
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Format data with the Llama chat template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")

def to_text(batch):
    texts = [tokenizer.apply_chat_template(m, tokenize = False, add_generation_prompt = False)
             for m in batch["messages"]]
    return {"text": texts}

dataset = dataset.map(to_text, batched = True)

split = dataset.train_test_split(test_size = 0.05, seed = 3407)
train_ds, eval_ds = split["train"], split["test"]
print(len(train_ds), "train /", len(eval_ds), "eval")
print(train_ds[0]["text"])

## 5. Train

- 1-2 epochs is plenty. Your examples overlap heavily (each one repeats earlier chat context), so more epochs mostly memorizes.
- `train_on_responses_only` makes the loss come from **your** replies only, not your friend's messages.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

EPOCHS = 1.5

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = EPOCHS,
        learning_rate = 1.4e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3411,
        output_dir = "outputs",
        report_to = "none",
        dataset_num_proc = 2,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
# Sanity check: the masked (ignored) parts should show as blank, your reply should be visible
sample = trainer.train_dataset[0]
print(tokenizer.decode(sample["input_ids"]))
print("-----")
print(tokenizer.decode([t for t in sample["labels"] if t != -100]))

In [ ]:
stats = trainer.train()
print(f"Peak VRAM: {torch.cuda.max_memory_reserved()/1e9:.1f} GB")

## 6. Chat with yourself

Paste recent chat lines in the same format as your training data. The model replies as **You**.
Lower `temperature` if it rambles; raise it if it's bland.

In [ ]:
FastLanguageModel.for_inference(model)

PREFIX = "Continue the WhatsApp conversation as You. Use the context below and respond naturally.\n\n"

def reply(context, temperature = 0.8, max_new_tokens = 140):
    messages = [{"role": "user", "content": PREFIX + context.strip()}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize = True, add_generation_prompt = True, return_tensors = "pt"
    ).to("cuda")
    out = model.generate(
        input_ids = inputs,
        max_new_tokens = max_new_tokens,
        do_sample = True,
        temperature = temperature,
        min_p = 0.1,
        use_cache = True,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens = True).strip()

context = '''[7/16/26, 6:59:22 PM] Friend: Hi! How can I help you?'''
for t in (0.6, 0.8, 1.0):
    print(f"--- temp {t} ---")
    print(reply(context, temperature = t))

## 7. Save the adapter and download it

In [ ]:
model.save_pretrained("lora_me")
tokenizer.save_pretrained("lora_me")

!zip -r lora_me.zip lora_me
from google.colab import files
files.download("lora_me.zip")

### Optional: export to GGUF for Ollama / llama.cpp
Uncomment to run (takes a few minutes):

In [ ]:
model.save_pretrained_gguf("me_gguf", tokenizer, quantization_method = "q4_k_m")
!zip -r me_gguf.zip me_gguf
files.download("me_gguf.zip")